# Notebook de EDA + Limpeza de dados Custos de Importação

## Configuração de Ambiente


### Bibliotecas python

In [255]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

### Caminho base do projeto

In [256]:
BASE_PATH = Path().resolve()

while BASE_PATH.name != "lh-nautical-data-project":
    BASE_PATH = BASE_PATH.parent

print(f"BASE PATH: {BASE_PATH}")

BASE PATH: /Users/richardgomes/lh-nautical-data-project


### Caminhos para os dados raw e staging

In [257]:
DATA_PATH = BASE_PATH / "data"
RAW_PATH = DATA_PATH / "raw"
STAGING_PATH = DATA_PATH / "staging"

In [258]:
print(f"RAW PATH: {RAW_PATH}")
print(f"STAGING PATH: {STAGING_PATH}")

RAW PATH: /Users/richardgomes/lh-nautical-data-project/data/raw
STAGING PATH: /Users/richardgomes/lh-nautical-data-project/data/staging


#### Leitura e carregamento dos dados dos custos de importação (custos_importacao.json)

In [259]:
with open(RAW_PATH / "custos_importacao.json", "r", encoding="utf-8") as f:
    custos_importacao_json = json.load(f)

In [260]:
custos_importacao = pd.json_normalize(custos_importacao_json)

In [261]:
custos_importacao.shape

(150, 4)

In [262]:
custos_importacao.head(10)

,product_id,product_name,category,historic_data
0,1,Transponder AIS Maré Magnum,eletrônicos,"[{'start_date': '10/08/2016', 'usd_price': 105..."
1,2,Transponder Furuno Marlin,eletrônicos,"[{'start_date': '23/11/2017', 'usd_price': 432..."
2,3,Radar Furuno Pulse Leviathan,eletrônicos,"[{'start_date': '12/04/2016', 'usd_price': 254..."
3,4,Rádio AIS Hydro Tidal Zen,eletrônicos,"[{'start_date': '04/03/2016', 'usd_price': 909..."
4,5,Piloto Automático Furuno Storm,eletrônicos,"[{'start_date': '10/02/2016', 'usd_price': 600..."
5,6,Transponder AIS Vector,eletrônicos,"[{'start_date': '23/07/2020', 'usd_price': 228..."
6,7,Radar AIS Zen,eletrônicos,"[{'start_date': '26/10/2016', 'usd_price': 625..."
7,8,GPS AIS Zen,eletrônicos,"[{'start_date': '06/12/2019', 'usd_price': 119..."
8,9,Transponder AIS Titan Pulse,eletrônicos,"[{'start_date': '26/12/2018', 'usd_price': 101..."
9,10,Piloto Automático Simrad Titan Flux Magnum,eletrônicos,"[{'start_date': '29/05/2018', 'usd_price': 859..."


Compreendendo o conteúdo de "historic_data"

In [263]:
custos_importacao["historic_data"].iloc[0]

[{'start_date': '10/08/2016', 'usd_price': 10583.63},
 {'start_date': '15/06/2018', 'usd_price': 8778.36},
 {'start_date': '25/09/2018', 'usd_price': 8023.87},
 {'start_date': '19/03/2019', 'usd_price': 8772.78},
 {'start_date': '17/01/2020', 'usd_price': 7918.18},
 {'start_date': '17/06/2020', 'usd_price': 6310.01},
 {'start_date': '02/07/2021', 'usd_price': 6586.7},
 {'start_date': '16/05/2022', 'usd_price': 6538.2},
 {'start_date': '28/02/2023', 'usd_price': 6360.91},
 {'start_date': '17/10/2023', 'usd_price': 6574.8},
 {'start_date': '16/02/2024', 'usd_price': 6657.12},
 {'start_date': '22/02/2024', 'usd_price': 6703.2},
 {'start_date': '15/03/2024', 'usd_price': 6633.66},
 {'start_date': '02/08/2024', 'usd_price': 5774.5},
 {'start_date': '08/04/2025', 'usd_price': 5579.75}]

Várias datas para apenas um único produto.
Vai ser preciso "explodir" a coluna

In [264]:
custos_importacao_exploded = custos_importacao.explode("historic_data")

custos_importacao_exploded.head()

,product_id,product_name,category,historic_data
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '10/08/2016', 'usd_price': 1058..."
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '15/06/2018', 'usd_price': 8778..."
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '25/09/2018', 'usd_price': 8023..."
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '19/03/2019', 'usd_price': 8772..."
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '17/01/2020', 'usd_price': 7918..."


Agora já dá pra separar em colunas, extraindo os campos do dicionário.

In [265]:
custos_importacao_exploded["start_date"] = custos_importacao_exploded["historic_data"].apply(lambda x: x["start_date"])

custos_importacao_exploded["usd_price"] = custos_importacao_exploded["historic_data"].apply(lambda x: x["usd_price"])

custos_importacao_exploded.head()

,product_id,product_name,category,historic_data,start_date,usd_price
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '10/08/2016', 'usd_price': 1058...",10/08/2016,10583.63
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '15/06/2018', 'usd_price': 8778...",15/06/2018,8778.36
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '25/09/2018', 'usd_price': 8023...",25/09/2018,8023.87
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '19/03/2019', 'usd_price': 8772...",19/03/2019,8772.78
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '17/01/2020', 'usd_price': 7918...",17/01/2020,7918.18


Agora posso montar as colunas finais.

In [266]:
custos_importacao_final = custos_importacao_exploded[[
    "product_id",
    "product_name",
    "category",
    "start_date",
    "usd_price"
]]

custos_importacao_final.head()

,product_id,product_name,category,start_date,usd_price
0,1,Transponder AIS Maré Magnum,eletrônicos,10/08/2016,10583.63
0,1,Transponder AIS Maré Magnum,eletrônicos,15/06/2018,8778.36
0,1,Transponder AIS Maré Magnum,eletrônicos,25/09/2018,8023.87
0,1,Transponder AIS Maré Magnum,eletrônicos,19/03/2019,8772.78
0,1,Transponder AIS Maré Magnum,eletrônicos,17/01/2020,7918.18


In [267]:
custos_importacao_final.info()

<class 'pandas.DataFrame'>
Index: 1260 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    1260 non-null   int64  
 1   product_name  1260 non-null   str    
 2   category      1260 non-null   str    
 3   start_date    1260 non-null   str    
 4   usd_price     1260 non-null   float64
dtypes: float64(1), int64(1), str(3)
memory usage: 59.1 KB


Tive que corrigir o tipo da data de "start_date"

In [268]:
custos_importacao_final["start_date"] = pd.to_datetime(
    custos_importacao_final["start_date"],
    dayfirst=True
)

custos_importacao_final.info()

<class 'pandas.DataFrame'>
Index: 1260 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   product_id    1260 non-null   int64         
 1   product_name  1260 non-null   str           
 2   category      1260 non-null   str           
 3   start_date    1260 non-null   datetime64[us]
 4   usd_price     1260 non-null   float64       
dtypes: datetime64[us](1), float64(1), int64(1), str(2)
memory usage: 59.1 KB


In [269]:
custos_importacao_final.head()

,product_id,product_name,category,start_date,usd_price
0,1,Transponder AIS Maré Magnum,eletrônicos,2016-08-10,10583.63
0,1,Transponder AIS Maré Magnum,eletrônicos,2018-06-15,8778.36
0,1,Transponder AIS Maré Magnum,eletrônicos,2018-09-25,8023.87
0,1,Transponder AIS Maré Magnum,eletrônicos,2019-03-19,8772.78
0,1,Transponder AIS Maré Magnum,eletrônicos,2020-01-17,7918.18


In [270]:
custos_importacao_exploded.head()

,product_id,product_name,category,historic_data,start_date,usd_price
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '10/08/2016', 'usd_price': 1058...",10/08/2016,10583.63
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '15/06/2018', 'usd_price': 8778...",15/06/2018,8778.36
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '25/09/2018', 'usd_price': 8023...",25/09/2018,8023.87
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '19/03/2019', 'usd_price': 8772...",19/03/2019,8772.78
0,1,Transponder AIS Maré Magnum,eletrônicos,"{'start_date': '17/01/2020', 'usd_price': 7918...",17/01/2020,7918.18


Removendo a coluna antiga "historic_data"

In [271]:
custos_importacao_exploded = custos_importacao_exploded.drop(columns=["historic_data"])

In [272]:
custos_importacao_exploded.head()

,product_id,product_name,category,start_date,usd_price
0,1,Transponder AIS Maré Magnum,eletrônicos,10/08/2016,10583.63
0,1,Transponder AIS Maré Magnum,eletrônicos,15/06/2018,8778.36
0,1,Transponder AIS Maré Magnum,eletrônicos,25/09/2018,8023.87
0,1,Transponder AIS Maré Magnum,eletrônicos,19/03/2019,8772.78
0,1,Transponder AIS Maré Magnum,eletrônicos,17/01/2020,7918.18


Padronizando os nomes

In [273]:
custos_importacao_exploded = custos_importacao_exploded.rename(columns={
    "product_id": "id_produto",
    "product_name": "nome_produto",
    "category": "categoria",
    "start_date": "data_inicio",
    "usd_price": "preco_usd"
})



Padronizando os tipos de dados: 

In [274]:
custos_importacao_exploded["nome_produto"] = custos_importacao_exploded["nome_produto"].str.title().str.strip()
custos_importacao_exploded["categoria"] = custos_importacao_exploded["categoria"].str.lower().str.strip()

Tipos numéricos

In [275]:
custos_importacao_exploded["id_produto"] = custos_importacao_exploded["id_produto"].astype("int64")
custos_importacao_exploded["preco_usd"] = custos_importacao_exploded["preco_usd"].astype("float64")

Tipo data (datetime)

In [276]:
custos_importacao_exploded["data_inicio"] = pd.to_datetime(
    custos_importacao_exploded["data_inicio"],
    dayfirst=True)

In [277]:
custos_importacao_exploded.info()

<class 'pandas.DataFrame'>
Index: 1260 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   id_produto    1260 non-null   int64         
 1   nome_produto  1260 non-null   str           
 2   categoria     1260 non-null   str           
 3   data_inicio   1260 non-null   datetime64[us]
 4   preco_usd     1260 non-null   float64       
dtypes: datetime64[us](1), float64(1), int64(1), str(2)
memory usage: 59.1 KB


In [278]:
custos_importacao_exploded.head()

,id_produto,nome_produto,categoria,data_inicio,preco_usd
0,1,Transponder Ais Maré Magnum,eletrônicos,2016-08-10,10583.63
0,1,Transponder Ais Maré Magnum,eletrônicos,2018-06-15,8778.36
0,1,Transponder Ais Maré Magnum,eletrônicos,2018-09-25,8023.87
0,1,Transponder Ais Maré Magnum,eletrônicos,2019-03-19,8772.78
0,1,Transponder Ais Maré Magnum,eletrônicos,2020-01-17,7918.18


Ordenação para o BI

In [279]:
custos_importacao_exploded = custos_importacao_exploded.sort_values(
    ["id_produto", "data_inicio"]
)

In [280]:
custos_importacao_exploded.head(10)

,id_produto,nome_produto,categoria,data_inicio,preco_usd
0,1,Transponder Ais Maré Magnum,eletrônicos,2016-08-10,10583.63
0,1,Transponder Ais Maré Magnum,eletrônicos,2018-06-15,8778.36
0,1,Transponder Ais Maré Magnum,eletrônicos,2018-09-25,8023.87
0,1,Transponder Ais Maré Magnum,eletrônicos,2019-03-19,8772.78
0,1,Transponder Ais Maré Magnum,eletrônicos,2020-01-17,7918.18
0,1,Transponder Ais Maré Magnum,eletrônicos,2020-06-17,6310.01
0,1,Transponder Ais Maré Magnum,eletrônicos,2021-07-02,6586.70
0,1,Transponder Ais Maré Magnum,eletrônicos,2022-05-16,6538.20
0,1,Transponder Ais Maré Magnum,eletrônicos,2023-02-28,6360.91
0,1,Transponder Ais Maré Magnum,eletrônicos,2023-10-17,6574.80


In [281]:
custos_importacao_exploded.to_csv(STAGING_PATH / "stg_custos_importacao_tratado.csv", index=False)

print("Arquivo salvo em:", STAGING_PATH / "stg_custos_importacao_tratado.csv")

Arquivo salvo em: /Users/richardgomes/lh-nautical-data-project/data/staging/stg_custos_importacao_tratado.csv
